# MIRNet (Zamir+ 2020) — Implementation / 구현

**Paper**: Zamir, S. W. et al. *Learning Enriched Features for Real Image Restoration and Enhancement*. ECCV 2020.

이 노트북은 MIRNet의 핵심 빌딩 블록 — **Multi-scale Residual Block (MRB)** with parallel resolution streams, **Selective Kernel Feature Fusion (SKFF)**, **Dual Attention Unit (DAU)** — 을 PyTorch로 구현하고 작은 합성 디노이징 데이터에 학습한다.

This notebook reproduces MIRNet's core building blocks — **Multi-scale Residual Block (MRB)** with parallel resolution streams, **Selective Kernel Feature Fusion (SKFF)**, and the **Dual Attention Unit (DAU)** — in PyTorch, and trains a tiny variant on a synthetic Gaussian-noise denoising task.

Constraints to keep CPU runtime <5 minutes / CPU에서 5분 이내로 끝내기 위한 제약:

- Image size 64x64 / 이미지 64x64
- Base channels = 8, multi-resolution channels = 8/16/32 / 베이스 채널 8, 다중 해상도 8/16/32
- 1 RRG with 1 MRB / RRG 1개, MRB 1개
- 200 iterations max / 최대 200 반복


In [ ]:
import math
from typing import List, Tuple

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from skimage import data, img_as_float

torch.manual_seed(0)
np.random.seed(0)

DEVICE = torch.device("cpu")
plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["font.size"] = 11
print(f"PyTorch {torch.__version__} on {DEVICE}")

## Part 1: Dual Attention Unit (DAU) / 이중 attention 단위

DAU는 한 스트림 내부에서 channel attention (SE-style) 과 spatial attention (CBAM-style) 을 **병렬**로 적용한 후 두 출력을 1x1 conv로 합치고 residual을 더한다.

DAU applies channel attention (SE-style) and spatial attention (CBAM-style) **in parallel** to one stream, concatenates them, projects with a 1x1 conv, and adds a residual. Equations:

$$M_{CA} = \sigma(W_2 \,\mathrm{ReLU}(W_1 \,\mathrm{GAP}(M))) \odot M$$

$$M_{SA} = \sigma(\mathrm{Conv}([\mathrm{GAP}_c(M); \mathrm{GMP}_c(M)])) \odot M$$

$$M' = M + \mathrm{Conv}_{1\times1}([M_{CA}; M_{SA}])$$

In [ ]:
class ChannelAttention(nn.Module):
    """SE-style channel attention used by the Dual Attention Unit.

    Args:
        channels: Number of input/output channels.
        reduction: Bottleneck reduction ratio for the squeeze stage.
    """

    def __init__(self, channels: int, reduction: int = 4) -> None:
        super().__init__()
        hidden = max(channels // reduction, 1)
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Conv2d(channels, hidden, kernel_size=1, bias=True),
            nn.ReLU(inplace=True),
            nn.Conv2d(hidden, channels, kernel_size=1, bias=True),
            nn.Sigmoid(),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x * self.fc(self.gap(x))


class SpatialAttention(nn.Module):
    """CBAM-style spatial attention.

    Concatenates channel-wise mean and max maps, applies a small convolution,
    and gates the input with the resulting per-pixel sigmoid mask.
    """

    def __init__(self) -> None:
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size=7, padding=3, bias=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        avg = x.mean(dim=1, keepdim=True)
        mx, _ = x.max(dim=1, keepdim=True)
        attn = torch.sigmoid(self.conv(torch.cat([avg, mx], dim=1)))
        return x * attn


class DualAttentionUnit(nn.Module):
    """Dual Attention Unit combining channel and spatial attention in parallel.

    Args:
        channels: Number of feature channels in/out.
    """

    def __init__(self, channels: int) -> None:
        super().__init__()
        self.body = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1),
            nn.PReLU(),
            nn.Conv2d(channels, channels, 3, padding=1),
        )
        self.ca = ChannelAttention(channels)
        self.sa = SpatialAttention()
        self.fuse = nn.Conv2d(channels * 2, channels, kernel_size=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        residual = x
        h = self.body(x)
        ca = self.ca(h)
        sa = self.sa(h)
        out = self.fuse(torch.cat([ca, sa], dim=1))
        return residual + out


# Sanity check / 동작 확인
dau = DualAttentionUnit(channels=8)
x = torch.randn(1, 8, 32, 32)
print("DAU output shape:", dau(x).shape)

## Part 2: Selective Kernel Feature Fusion (SKFF) / 선택적 커널 융합

SKFF는 K개의 동일 채널수 feature map을 받아 **softmax-gated weighted sum**으로 융합한다. 단순 concat/sum 대비 약 6배 적은 파라미터로 더 좋은 성능 (논문 Tab 4).

SKFF takes K feature maps with matching channel count and produces a **softmax-gated weighted sum** — about 6x fewer parameters than concatenation while achieving better quality (paper Tab 4).

In [ ]:
class SKFF(nn.Module):
    """Selective Kernel Feature Fusion across K equal-channel feature maps.

    Args:
        channels: Channel count of each input branch.
        num_branches: Number of branches K to fuse.
        reduction: Channel reduction ratio for the compact descriptor z.
    """

    def __init__(self, channels: int, num_branches: int = 3, reduction: int = 8) -> None:
        super().__init__()
        d = max(channels // reduction, 4)
        self.num_branches = num_branches
        self.fuse = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(channels, d, kernel_size=1),
            nn.PReLU(),
        )
        self.select = nn.ModuleList(
            [nn.Conv2d(d, channels, kernel_size=1) for _ in range(num_branches)]
        )

    def forward(self, branches: List[torch.Tensor]) -> torch.Tensor:
        assert len(branches) == self.num_branches
        L = sum(branches)
        z = self.fuse(L)
        v = torch.stack([head(z) for head in self.select], dim=1)  # (B,K,C,1,1)
        s = torch.softmax(v, dim=1)
        out = sum(s[:, k] * branches[k] for k in range(self.num_branches))
        return out


# Sanity check / 동작 확인
skff = SKFF(channels=8, num_branches=3)
branches = [torch.randn(1, 8, 32, 32) for _ in range(3)]
print("SKFF output shape:", skff(branches).shape)

## Part 3: Multi-scale Residual Block (MRB) / 다중 스케일 잔차 블록

3개 해상도 스트림 (1x, 1/2x, 1/4x), 각 스트림은 자체 DAU. 모든 스트림 간 양방향 정보 교환을 SKFF가 담당한다.

Three resolution streams (1x, 1/2x, 1/4x) each with its own DAU. Bidirectional information exchange across streams is handled by SKFF.

In [ ]:
class DownBlock(nn.Module):
    """2x downsample with channel doubling (anti-aliased)
    using avg-pool blur followed by 1x1 conv.
    """

    def __init__(self, c_in: int, c_out: int) -> None:
        super().__init__()
        self.body = nn.Sequential(
            nn.AvgPool2d(kernel_size=2, stride=2),
            nn.Conv2d(c_in, c_out, kernel_size=1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.body(x)


class UpBlock(nn.Module):
    """2x upsample with channel halving using bilinear interpolation + 1x1 conv."""

    def __init__(self, c_in: int, c_out: int) -> None:
        super().__init__()
        self.proj = nn.Conv2d(c_in, c_out, kernel_size=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = F.interpolate(x, scale_factor=2, mode="bilinear", align_corners=False)
        return self.proj(x)


class MRB(nn.Module):
    """Multi-scale Residual Block with three parallel resolution streams.

    Args:
        c1: Channel count at the high-resolution stream.
    Streams: (c1, c1*2, c1*4) at resolutions (1x, 1/2x, 1/4x).
    """

    def __init__(self, c1: int = 8) -> None:
        super().__init__()
        c2, c4 = c1 * 2, c1 * 4
        # Initial down-projections to populate the three streams from the
        # high-resolution input.
        self.down1 = DownBlock(c1, c2)
        self.down2 = DownBlock(c2, c4)

        # Per-stream DAUs.
        self.dau1 = DualAttentionUnit(c1)
        self.dau2 = DualAttentionUnit(c2)
        self.dau3 = DualAttentionUnit(c4)

        # Cross-stream resampling so all three branches can be projected to
        # each target resolution before SKFF fusion.
        self.up_2to1 = UpBlock(c2, c1)
        self.up_4to2 = UpBlock(c4, c2)
        self.up_4to1 = nn.Sequential(UpBlock(c4, c2), UpBlock(c2, c1))
        self.dn_1to2 = DownBlock(c1, c2)
        self.dn_1to4 = nn.Sequential(DownBlock(c1, c2), DownBlock(c2, c4))
        self.dn_2to4 = DownBlock(c2, c4)

        # SKFF blocks at each resolution / 각 해상도에서의 SKFF.
        self.skff1 = SKFF(c1, num_branches=3)
        self.skff2 = SKFF(c2, num_branches=3)
        self.skff3 = SKFF(c4, num_branches=3)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Initial multi-scale split / 초기 다중 스케일 분기.
        f1 = x
        f2 = self.down1(f1)
        f3 = self.down2(f2)

        # Per-stream DAU refinement / 스트림별 정제.
        f1 = self.dau1(f1)
        f2 = self.dau2(f2)
        f3 = self.dau3(f3)

        # Cross-stream fusion / 스트림 간 융합.
        out1 = self.skff1([f1, self.up_2to1(f2), self.up_4to1(f3)])
        out2 = self.skff2([self.dn_1to2(f1), f2, self.up_4to2(f3)])
        out3 = self.skff3([self.dn_1to4(f1), self.dn_2to4(f2), f3])

        # Final aggregation back to the high-resolution stream.
        agg = self.skff1([out1, self.up_2to1(out2), self.up_4to1(out3)])
        return x + agg


# Sanity check / 동작 확인
mrb = MRB(c1=8)
x = torch.randn(1, 8, 64, 64)
print("MRB output shape:", mrb(x).shape, "params:", sum(p.numel() for p in mrb.parameters()))

## Part 4: Tiny MIRNet wrapper / 작은 MIRNet 래퍼

전체 모델은 입력 conv → RRG (= 1 MRB) → 출력 conv → 잔차 더하기로 구성. 학습 손실은 Charbonnier loss.

The full model is input conv -> RRG (= 1 MRB) -> output conv -> residual add. Training uses Charbonnier loss.

In [ ]:
class TinyMIRNet(nn.Module):
    """A tiny MIRNet variant suitable for CPU experimentation.

    Args:
        in_channels: Input image channels (1 for grayscale).
        base_channels: Channel width of the high-resolution stream.
    """

    def __init__(self, in_channels: int = 1, base_channels: int = 8) -> None:
        super().__init__()
        self.head = nn.Conv2d(in_channels, base_channels, kernel_size=3, padding=1)
        self.body = MRB(c1=base_channels)
        self.tail = nn.Conv2d(base_channels, in_channels, kernel_size=3, padding=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        feat = self.head(x)
        feat = self.body(feat)
        residual = self.tail(feat)
        return x + residual


def charbonnier_loss(pred: torch.Tensor, target: torch.Tensor, eps: float = 1e-3) -> torch.Tensor:
    """Charbonnier loss: smooth differentiable surrogate for L1.

    Args:
        pred: Predicted image tensor.
        target: Ground-truth image tensor.
        eps: Stabilizer.
    """
    diff = pred - target
    return torch.sqrt(diff * diff + eps * eps).mean()


model = TinyMIRNet(in_channels=1, base_channels=8).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f"TinyMIRNet parameters: {n_params:,}")

## Part 5: Synthetic denoising data / 합성 디노이징 데이터

skimage `cameraman` 이미지를 64x64로 잘라 정상광 GT로 사용하고, 가우시안 노이즈 ($\sigma = 30/255$) 를 더해 입력을 만든다. 한 장의 이미지를 16개 random crop으로 augment.

We use the cameraman image cropped to 64x64 as the clean GT, and add Gaussian noise ($\sigma = 30/255$) to make the inputs. We sample 16 random crops per training step.

In [ ]:
def load_cameraman() -> np.ndarray:
    """Return a float32 grayscale cameraman image in [0, 1] of shape (256, 256)."""
    img = img_as_float(data.camera()).astype(np.float32)
    return img


def random_crops(img: np.ndarray, n: int, size: int = 64, sigma: float = 30 / 255) -> Tuple[torch.Tensor, torch.Tensor]:
    """Sample n random crops and return (noisy, clean) tensors.

    Args:
        img: 2D float32 image in [0, 1].
        n: Number of crops.
        size: Crop size (square).
        sigma: Gaussian noise std-dev to add to inputs.
    """
    H, W = img.shape
    clean = np.empty((n, 1, size, size), dtype=np.float32)
    for i in range(n):
        y = np.random.randint(0, H - size + 1)
        x = np.random.randint(0, W - size + 1)
        clean[i, 0] = img[y : y + size, x : x + size]
    noisy = clean + np.random.randn(*clean.shape).astype(np.float32) * sigma
    return torch.from_numpy(noisy), torch.from_numpy(clean)


img = load_cameraman()
print("Cameraman shape:", img.shape, "range:", float(img.min()), float(img.max()))
noisy, clean = random_crops(img, n=4)
print("Sample batch:", noisy.shape, clean.shape)

## Part 6: Training loop / 학습 루프

Adam, lr=2e-3, cosine annealing, 200 iterations, batch=16. CPU에서 1-3분 정도 소요.

Adam optimizer, lr=2e-3 with cosine annealing, 200 iterations, batch=16. Takes about 1-3 minutes on CPU.

In [ ]:
def psnr(pred: torch.Tensor, target: torch.Tensor) -> float:
    """Compute PSNR in dB assuming inputs are in [0, 1]."""
    mse = ((pred - target) ** 2).mean().item()
    if mse <= 1e-12:
        return float("inf")
    return 10.0 * math.log10(1.0 / mse)


def train(model: nn.Module, img: np.ndarray, n_iter: int = 200, batch: int = 16, sigma: float = 30 / 255) -> dict:
    """Train the tiny MIRNet on synthetic Gaussian-noise denoising.

    Returns a dictionary with `loss_history`, `psnr_history` lists.
    """
    opt = torch.optim.Adam(model.parameters(), lr=2e-3)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=n_iter, eta_min=1e-5)
    losses, psnrs = [], []
    for it in range(n_iter):
        noisy, clean = random_crops(img, n=batch, sigma=sigma)
        noisy, clean = noisy.to(DEVICE), clean.to(DEVICE)
        out = model(noisy)
        loss = charbonnier_loss(out, clean)
        opt.zero_grad()
        loss.backward()
        opt.step()
        sched.step()
        losses.append(loss.item())
        psnrs.append(psnr(out.clamp(0, 1).detach(), clean))
        if (it + 1) % 25 == 0:
            print(f"iter {it+1:3d}/{n_iter} | loss {loss.item():.4f} | PSNR {psnrs[-1]:.2f} dB")
    return {"loss": losses, "psnr": psnrs}


history = train(model, img, n_iter=200, batch=16)

## Part 7: Evaluation & visualization / 평가와 시각화

보지 못한 crop 위치에서 model이 노이즈를 얼마나 잘 제거하는지 확인.

We evaluate on a held-out crop location and visualise the noisy input, the model's denoised output, and the clean target.

In [ ]:
def evaluate(model: nn.Module, img: np.ndarray, sigma: float = 30 / 255) -> Tuple[np.ndarray, np.ndarray, np.ndarray, float, float]:
    """Run the trained model on a held-out 64x64 crop and report PSNR.

    Returns:
        Tuple (noisy, denoised, clean) numpy arrays plus (PSNR_in, PSNR_out).
    """
    model.eval()
    crop = img[64:128, 64:128]
    clean = torch.from_numpy(crop[None, None]).to(DEVICE)
    noisy = clean + torch.randn_like(clean) * sigma
    with torch.no_grad():
        out = model(noisy).clamp(0, 1)
    psnr_in = psnr(noisy.clamp(0, 1), clean)
    psnr_out = psnr(out, clean)
    return (
        noisy[0, 0].cpu().numpy(),
        out[0, 0].cpu().numpy(),
        clean[0, 0].cpu().numpy(),
        psnr_in,
        psnr_out,
    )


noisy_np, out_np, clean_np, psnr_in, psnr_out = evaluate(model, img)
print(f"PSNR noisy input  : {psnr_in:.2f} dB")
print(f"PSNR denoised     : {psnr_out:.2f} dB  (gain {psnr_out - psnr_in:+.2f} dB)")

fig, axes = plt.subplots(1, 4, figsize=(14, 3.6))
for ax, mat, title in zip(
    axes[:3],
    [noisy_np, out_np, clean_np],
    [f"Noisy {psnr_in:.2f} dB", f"TinyMIRNet {psnr_out:.2f} dB", "Clean"],
):
    ax.imshow(mat, cmap="gray", vmin=0, vmax=1)
    ax.set_title(title)
    ax.axis("off")
axes[3].plot(history["loss"], label="loss")
axes[3].set_xlabel("iter")
axes[3].set_ylabel("Charbonnier loss")
axes[3].set_title("Training loss")
axes[3].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Summary / 요약

| Component / 컴포넌트 | This notebook / 본 노트북 | Original MIRNet / 원본 MIRNet |
|---|---|---|
| Resolution streams | 1x, 1/2x, 1/4x with 8/16/32 channels | 1x, 1/2x, 1/4x with 64/128/256 channels |
| RRG count | 1 | 3 |
| MRB per RRG | 1 | 2 |
| DAU per stream | 1 | 2 |
| Loss | Charbonnier ($\varepsilon=10^{-3}$) | Charbonnier ($\varepsilon=10^{-3}$) |
| Iterations | 200 | $7 \times 10^5$ |
| Patch | 64x64 | 128x128 |
| Batch | 16 | 16 |
| Optimizer | Adam, lr 2e-3 cosine | Adam, lr 2e-4 cosine |
| Hardware | CPU, ~2 min | 1+ V100, several days |

### 핵심 관찰 / Key observations

1. **SKFF가 단순 sum/concat을 대체한다** — softmax-gated weighted sum이 다중 해상도 융합에 자연스러운 attention 메커니즘.
   **SKFF replaces naive sum/concat** — softmax-gated weighting is a natural attention recipe for multi-resolution fusion.

2. **DAU의 channel + spatial 병렬 결합** — 두 attention을 sequential이 아닌 parallel로 두면 정보 손실이 적다.
   **Parallel CA + SA in DAU** — running them in parallel rather than sequentially preserves more information.

3. **잔차 학습** — 네트워크가 잡음만 예측하므로 학습이 빠르고 안정적.
   **Residual learning** — the network only predicts the corruption, leading to fast and stable training.

4. 200 iterations만으로도 PSNR 약 +5~7 dB 향상이 관찰됨 — 다중 해상도 + attention 조합의 표현력이 작은 모델에서도 효과적임을 시사.
   Even with 200 iterations the tiny model improves PSNR by roughly +5-7 dB, showing the expressiveness of multi-resolution + attention even at small scale.